# Evaluating Survey Area for various survey modes, in baseline survey v5.3 #

- creation date : 2026-10-07
- notebook on survey stragegy : https://github.com/lsst/survey-strategy_lsst_io/blob/main/notebooks/SurveyAreas.ipynb
- conda install -c conda-forge tabulate

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import colorcet as cc
import pandas as pd
import healpy as hp
from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
nside = 64

# get_current_footprint returns the 'target map' weightings over the sky per filter,
# and the 'labels' indicating the primary survey purpose in that area
# note that DDF and other microsurveys get added in the FBS, so are not included in these maps

skymap = CurrentAreaMap(nside=nside)  # , dust_limit=0.12)
maps, labels = skymap.return_maps()

In [ ]:
wfd = np.where(
    (labels == "lowdust")
    | (labels == "euclid_overlap")
    | (labels == "bulgy")
    | (labels == "virgo")
    | (labels == "LMC_SMC"),
    1,
    0,
)
alpha = np.where(wfd == 1, 1, 0.7)
alpha = np.where((skymap.dustmap < 0.12) & (maps["r"] > 0), alpha, alpha - 0.3)
hp.mollview(skymap.dustmap, min=0, max=1.2, alpha=alpha, title=None, cbar=False)
hp.graticule()

In [ ]:
# what labels do we have?
label_names = [l for l in np.unique(labels) if l != ""]
label_names

In [ ]:
# let's show them on the sky

label_dict = {}
for i, name in enumerate(label_names):
    label_dict[name] = i + 1
label_dict

In [ ]:
ft = np.zeros(len(maps["r"]))
for name in label_names:
    match = np.where(labels == name)
    ft[match] += label_dict[name]
ft = np.round(ft).astype(int)
ft = np.where(ft == 0, np.nan, ft)

glasbey_cmap = ListedColormap(cc.glasbey_category10, name="glasbey_category10")

# Pass the colormap object to projview
_ = hp.newvisufunc.projview(ft, cmap=glasbey_cmap, graticule=True, cbar=False)

In [ ]:
# Area on-sky in each of these regions?
pixarea = hp.nside2pixarea(nside, degrees=True)
# fudge factor to cover X amount of area
# overlap between pointings in the tesselation is 10%,
# but edge effects of the footprint increases the needed
# number of visits (to fully cover the area, as some visits 'hang over' the edge)
# NOTE THIS IS APPROXIMATE and will vary with footprint details
pixarea_to_pointings = (1 / 9.6) * 1.3

skarea = {}
skapprox_pointings = {}
for name in label_names:
    npix = len(np.where(labels == name)[0])
    skarea[name] = npix * pixarea
    skapprox_pointings[name] = np.ceil(skarea[name] * pixarea_to_pointings)
    # this would be 1.1 for a 10% overlap between pointings, but
# allfootprint
npix = len(np.where(labels != "")[0])
skarea["All"] = npix * pixarea
skapprox_pointings["All"] = np.ceil(skarea["All"] * pixarea_to_pointings)

print("Area per SKY_AREA section")
pd.DataFrame([skarea, skapprox_pointings], index=["Area (sq deg)", "Approx NPointing"]).round(1).T

In [ ]:
# how many visits in the current baseline fall into each of these regions?
from rubin_sim.data import get_baseline
import sqlite3
from rubin_sim.maf import WFDlabelStacker
import rubin_sim.maf as maf

# opsdb = get_baseline()
# opsdb = "baseline_v5.3.6_10yrs.db"
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
print(opsdb)
conn = sqlite3.connect(opsdb)
opsim = os.path.split(opsdb)[-1].replace(".db.", "")
print(opsim, ":", opsdb)
visits = pd.read_sql(
    "select observationStartMJD, band, fieldRA, fieldDec, rotSkyPos, visitExposureTime, target_name, scheduler_note from observations",
    conn,
)

visits.head()

In [ ]:
# Microsurveys - count up total visits by labelling directly
twilight_neo = visits.query("scheduler_note.str.contains('twilight_near_sun')")
ddf = visits.query("scheduler_note.str.contains('DD') and not scheduler_note.str.contains('RGES')")
rges = visits.query("scheduler_note.str.contains('RGES')")
triplets = visits.query("scheduler_note.str.contains('long') and not scheduler_note.str.contains('blob')")
nvis = {}
nvis["all visits"] = len(visits)
nvis["TwilightNEO"] = len(twilight_neo)
nvis["DD"] = len(ddf)
nvis["Triplet (third)"] = len(triplets)
nvis["RGES"] = len(rges)
nvis

In [ ]:
area = {}
approx_pointings = {}
med_nvis = {}
mean_nvis = {}

plot_dict = {"percentile_clip": 95}

name = "all visits"
print(f"Plotting for {name}")
vis = visits.query("not scheduler_note.str.contains('ToO')").to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "DD"
print(f"Plotting for {name}")
vis = ddf.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "RGES"
print(f"Plotting for {name}")
vis = rges.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "TwilightNEO"
print(f"Plotting for {name}")
vis = twilight_neo.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

name = "Triplet (third)"
print(f"Plotting for {name}")
vis = triplets.to_records()
if len(vis) > 0:
    metric = maf.CountMetric(col="observationStartMJD")
    slicer = maf.HealpixSlicer(nside=nside)
    summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
    b = maf.MetricBundle(
        metric,
        slicer,
        constraint="",
        summary_metrics=summaries,
        run_name=opsim,
        info_label=name,
        plot_dict=plot_dict,
    )
    g = maf.MetricBundleGroup({0: b}, None)
    g.run_current(sim_data=vis, constraint="")

    area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
    approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
    med_nvis[name] = b.summary_values["Median"]
    mean_nvis[name] = b.summary_values["Mean"]

    print(b.summary_values)
    figs = b.plot()

In [ ]:
other_visits = visits.query(
    "~scheduler_note.str.contains('DD') and ~scheduler_note.str.contains('twilight_near_sun') and ~scheduler_note.str.contains('ToO')"
)
ovisitsnp = other_visits.to_records()

In [ ]:
footpts = {}

In [ ]:
# Visits can count towards more than one area, but there will be a minimum number of visits required to cover a given area
# The WFDlabelStacker can "attribute" a visit towards a given area --
# the threshold is if >0.4 of the visit area falls into the "footpt" healpix than it is counted toward covering that area.

name = "NES"
footpts[name] = np.where(labels == "nes", 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "SCP"
footpts[name] = np.where(labels == "scp", 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "MW"
footpts[name] = np.where((labels == "bulgy") | (labels == "dusty_plane") | (labels == "LMC_SMC"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "MW non-WFD"
footpts[name] = np.where((labels == "dusty_plane"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "ExGal"
footpts[name] = np.where((labels == "lowdust") | (labels == "euclid_overlap"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "Dust<0.12"
footpts[name] = np.where((skymap.dustmap < 0.12) & (labels == "lowdust"), 1, 0)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
name = "WFD"
footpts[name] = np.where(
    (labels == "lowdust")
    | (labels == "euclid_overlap")
    | (labels == "bulgy")
    | (labels == "virgo")
    | (labels == "LMC_SMC"),
    1,
    0,
)
stacker = WFDlabelStacker(footprint=footpts[name], area_id_name=name)
ovisitsnp = stacker.run(sim_data=ovisitsnp)
nvis[name] = len(np.where(ovisitsnp["area_id"] == name)[0])

print(f"Plotting for {name}")
vis = ovisitsnp[np.where(ovisitsnp["area_id"] == name)]
metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]
b = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({0: b}, None)
g.run_current(sim_data=vis, constraint="")

area[name] = len(np.where(b.metric_values.filled(0) > 0)[0]) * pixarea
approx_pointings[name] = np.ceil(area[name] * pixarea_to_pointings)
med_nvis[name] = b.summary_values["Median"]
mean_nvis[name] = b.summary_values["Mean"]

print(b.summary_values)
figs = b.plot()

In [ ]:
# Totalling up based on *visits required to cover a given region (or for a particular microsurvey)*
fraction = dict(zip(list(nvis.keys()), np.array(list(nvis.values())) / nvis["all visits"] * 100))
fraction["TwilightNEO"] /= 2  # visits are half as long

nvis["ToO"] = len(visits.query("scheduler_note.str.contains('ToO')"))
fraction["ToO"] = nvis["ToO"] / nvis["all visits"] * 100
pd.DataFrame(
    [nvis, fraction, area, approx_pointings, med_nvis, mean_nvis],
    index=["Nvisits", "Percent", "Area", "Approx NPointings", "Median Nvis", "Mean Nvis"],
).round(1).T

In [ ]:
# The methods above were the only way to count up the total number of visits which would be required to completely cover each region.
# However, we can get an easier estimate of the number of visits per point *within* each region by just calculating over the whole sky
# And then subsetting the resulting healpix map.

other_visits = visits.query(
    "~scheduler_note.str.contains('DD') and ~scheduler_note.str.contains('twilight_near_sun') and ~scheduler_note.str.contains('ToO')"
)

metric = maf.CountMetric(col="observationStartMJD")
slicer = maf.HealpixSlicer(nside=nside)
summaries = [maf.MeanMetric(), maf.MedianMetric(), maf.RmsMetric()]

bundles = {}
for band in "ugrizy":
    vis = other_visits.query("band == @band").to_records()
    bundles[band] = maf.MetricBundle(
        metric, slicer, constraint=f"{band} band", summary_metrics=summaries, run_name=opsim, info_label=name
    )
    g = maf.MetricBundleGroup({band: bundles[band]}, None)
    g.run_current(sim_data=vis, constraint=f"{band} band")

vis = other_visits.to_records()
bundles["all"] = maf.MetricBundle(
    metric, slicer, constraint="", summary_metrics=summaries, run_name=opsim, info_label=name
)
g = maf.MetricBundleGroup({"all": bundles["all"]}, None)
g.run_current(sim_data=vis, constraint="")

In [ ]:
median_stats = {}
mean_stats = {}
for name in footpts:
    mean_stats[name] = {}
    median_stats[name] = {}
    mask = np.where(footpts[name] > 0)[0]
    for band in "ugrizy":
        mval = bundles[band].metric_values.filled(0)[mask]
        median_stats[name][band] = np.median(mval)
        mean_stats[name][band] = np.mean(mval)
    mval = bundles["all"].metric_values.filled(0)[mask]
    median_stats[name]["all"] = np.median(mval)
    mean_stats[name]["all"] = np.mean(mval)

In [ ]:
dd = pd.DataFrame(median_stats).T.astype(int)
dd.columns = [f"med {b}" for b in dd.columns]
dd

In [ ]:
dd2 = pd.DataFrame(median_stats).T.astype(int)
dd2.columns = [f"mean {b}" for b in dd2.columns]
dd2

In [ ]:
dd.merge(dd2, right_index=True, left_index=True)